# Theorical Beam Divergence Comparison Between Focused And Unfocused Transducers

## Unfocused

In [10]:
import math

def calculate_sampled_volume(z, c_cfrp, f_nominal, D, n_cycles, anisotropy_factor=1.0, f_downshifted=None):
    """
    Computes the sampled volume of a single A-scan in a CFRP sample.
    
    Parameters:
    z (float): Depth of interest in the CFRP sample (mm)
    c_cfrp (float): Longitudinal wave velocity in the specific CFRP layup (mm/us)
    f_nominal (float): Nominal transducer frequency (MHz)
    D (float): Transducer element diameter (mm)
    n_cycles (float): Number of cycles in the broadband pulse
    anisotropy_factor (float): Factor to scale beam width due to CFRP anisotropy (default 1.0)
    f_downshifted (float): Measured downshifted frequency at depth z (MHz). 
                           If None, f_nominal is used.
    
    Returns:
    tuple: Sampled volume (mm^3), Axial resolution (mm), Beam diameter (mm), Effective Frequency (MHz)
    """
    
    # 1. Frequency Downshifting Correction
    # Uses the downshifted frequency if provided, otherwise defaults to nominal frequency.
    f_eff = f_downshifted if f_downshifted else f_nominal
    
    # 2. Calculate Axial Resolution
    # Equation: R_axial = (n * c) / (2 * f)
    R_axial = (n_cycles * c_cfrp) / (2.0 * f_eff)
    
    # 3. Calculate Lateral Resolution (-6 dB Beam Diameter)
    
    # NEW: The beam has a finite diameter at the end of the near zone (frontwall where z=0)
    # Equation for flat transducer at natural focus: BD_start = 0.2568 * D
    BD_start = 0.2568 * D
    
    # Beam spread half-angle (gamma) for -6 dB drop 
    # Equation: sin(gamma) = 0.514 * c / (f * D)
    sin_gamma = 0.514 * c_cfrp / (f_eff * D)
    
    # Safety check
    if sin_gamma >= 1.0:
        raise ValueError("Beam spread angle calculation exceeds 90 degrees.")
        
    gamma = math.asin(sin_gamma)
    
    # The beam diameter at depth z is the starting diameter plus the far-field spread
    # scaled by the CFRP anisotropy factor.
    BD_6dB = BD_start + (2.0 * z * math.tan(gamma) * anisotropy_factor)
    
    # 4. Calculate Radius, Lateral Area, and Sampled Volume
    radius = BD_6dB / 2.0
    lateral_area = math.pi * (radius ** 2) 
    V_sampled = lateral_area * R_axial
    
    return V_sampled, R_axial, BD_6dB, f_eff, lateral_area, radius, BD_start

In [26]:
# Define parameters based on a typical 5 MHz transducer inspecting a CFRP part
depth_z = 5              # Depth of interest: 10 mm
velocity_cfrp = 2.97          # Measured local velocity for CFRP: 2.9 mm/us 
frequency_nom = 5.0          # Transducer nominal frequency: 5.0 MHz 
diameter_D = 9.525            # Transducer diameter: 9.525 mm 
pulse_cycles = 1           # Highly damped pulse: 2 cycles 

# CFRP-Specific Corrections
aniso_factor = 1           # Example scaling factor for beam distortion in this layup 
freq_downshifted = 3       # Effective frequency drops to 4 MHz at 10 mm depth due to a

vol, r_ax, bd, f_effective, lateral_area, radius, bd_start = calculate_sampled_volume(
        z=depth_z, 
        c_cfrp=velocity_cfrp, 
        f_nominal=frequency_nom, 
        D=diameter_D, 
        n_cycles=pulse_cycles, 
        anisotropy_factor=aniso_factor, 
        f_downshifted=freq_downshifted
    )

# Output results
print(f"--- Ultrasound Sampled Volume in CFRP ---")
print(f"Depth of interest (z): {depth_z} mm")
print(f"Effective Frequency:   {f_effective} MHz")
print(f"Axial Resolution:      {r_ax:.3f} mm")
print(f"-6 dB Beam Diameter:   {bd:.3f} mm")
print(f"Sampled Volume:        {vol:.3f} mm^3")
print(f"Lateral Area:          {lateral_area:.3f} mm^2")
print(f"Radius:                {radius:.3f} mm")
print(f"Starting Beam Diameter: {bd_start:.3f} mm")

--- Ultrasound Sampled Volume in CFRP ---
Depth of interest (z): 5 mm
Effective Frequency:   3 MHz
Axial Resolution:      0.495 mm
-6 dB Beam Diameter:   2.981 mm
Sampled Volume:        3.455 mm^3
Lateral Area:          6.979 mm^2
Radius:                1.491 mm
Starting Beam Diameter: 2.446 mm


## Focused

In [28]:
import math

def calculate_focused_beam_diameter(z, F_w, WP, D, f_eff, c_water, c_cfrp, anisotropy_factor=1.0):
    """
    Calculates the -6 dB beam diameter of a focused immersion transducer
    at a specific depth inside a CFRP sample.
    """
    # 1. Focal depth inside CFRP (Focal Shift)
    # If the water path equals the water focal length, it focuses at the frontwall (z_s = 0)
    if WP >= F_w:
        z_s = 0.0
    else:
        z_s = (F_w - WP) * (c_cfrp / c_water)

    # 2. Convergence half-angle in water
    # Based on transducer geometry
    gamma_water = math.atan((D / 2.0) / F_w)

    # 3. Refracted divergence half-angle in CFRP (Snell's Law)
    # Because sound is faster in CFRP, the angle bends outward (diverges faster)
    sin_gamma_cfrp = (c_cfrp / c_water) * math.sin(gamma_water)
    if sin_gamma_cfrp >= 1.0:
        raise ValueError("Total internal reflection. Beam will not penetrate CFRP.")
    gamma_cfrp = math.asin(sin_gamma_cfrp)

    # 4. Minimum Beam Diameter at the focal point
    # Equation for focused -6 dB beam diameter: BD = 1.028 * c * F / (f * D)
    if z_s == 0.0:
        # If focused exactly on the frontwall, it is limited by the water focal spot size
        BD_focus = (1.028 * c_water * F_w) / (f_eff * D)
    else:
        # If focused inside the CFRP, it is based on the shifted focal depth
        BD_focus = (1.028 * c_cfrp * z_s) / (f_eff * D)

    # 5. Beam Diameter at Depth z
    # The beam diverges as a cone from the focal depth z_s
    divergence_spread = 2.0 * abs(z - z_s) * math.tan(gamma_cfrp)
    
    # Apply CFRP anisotropy factor (accounts for excess beam divergence and distortion)
    BD_z = (BD_focus + divergence_spread) * anisotropy_factor

    return BD_z, z_s, math.degrees(gamma_cfrp)


In [31]:
# Common inspection parameters
depth_z = 5               # Inspecting a flaw 10 mm deep into the CFRP
c_water = 1.5              # Velocity in water (mm/us)
c_cfrp = 2.97                 # Velocity in CFRP (mm/us)
D = 9.525                    # Transducer diameter (mm)
f_eff = 3.0                  # Effective frequency (MHz)
aniso = 1                  # CFRP anisotropy factor

print(f"--- BEAM DIAMETER AT {depth_z} mm DEPTH ---\n")

# 1. FLAT TRANSDUCER (Near zone placed at frontwall)
BD_start_flat = 0.2568 * D
gamma_flat = math.asin(0.514 * c_cfrp / (f_eff * D))
BD_flat = BD_start_flat + (2.0 * depth_z * math.tan(gamma_flat) * aniso)

print(f"FLAT TRANSDUCER:")
print(f"Divergence Half-Angle: {math.degrees(gamma_flat):.2f} degrees")
print(f"Beam Diameter at {depth_z}mm: {BD_flat:.3f} mm\n")

# 2. FOCUSED TRANSDUCER (Focal length placed at frontwall, WP = F_w)
F_w = 50.0 # 50 mm focal length in water
WP = 50.0  # Water path exactly equals focal length

BD_focused, z_s, angle_focused = calculate_focused_beam_diameter(
    depth_z, F_w, WP, D, f_eff, c_water, c_cfrp, aniso
)

print(f"FOCUSED TRANSDUCER (Focused on Frontwall):")
print(f"Refracted Divergence Half-Angle: {angle_focused:.2f} degrees")
print(f"Beam Diameter at {depth_z}mm: {BD_focused:.3f} mm")

--- BEAM DIAMETER AT 5 mm DEPTH ---

FLAT TRANSDUCER:
Divergence Half-Angle: 3.06 degrees
Beam Diameter at 5mm: 2.981 mm

FOCUSED TRANSDUCER (Focused on Frontwall):
Refracted Divergence Half-Angle: 10.82 degrees
Beam Diameter at 5mm: 4.610 mm
